In [95]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report





In [96]:
df = pd.read_csv("../data/processed/diabetes_clean_2.csv")
df.shape

(70422, 30)

In [97]:
X = df.drop("readmitted", axis=1)
y = df["readmitted"]

In [100]:
numeric_features = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses'
]

categorical_features = [
    'race','gender','age','metformin','repaglinide','nateglinide',
    'glimepiride','glipizide','glyburide','pioglitazone','rosiglitazone',
    'insulin','glyburide-metformin','change','diabetesMed',
    'diag_1_grp','diag_2_grp','diag_3_grp',
    'admission_type_id','discharge_disposition_id','admission_source_id'
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [101]:
# split first
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (56337, 29)
X_test: (14085, 29)
y_train: (56337,)
y_test: (14085,)


In [102]:

# pipeline
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])
# train
lr_pipeline.fit(X_train, y_train)

# predict
y_pred = lr_pipeline.predict(X_test)

# evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.6665246716364928
Precision: 0.1385326886693294
Recall: 0.5238473767885533
F1-score: 0.21911886949293433

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.68      0.79     12827
           1       0.14      0.52      0.22      1258

    accuracy                           0.67     14085
   macro avg       0.54      0.60      0.50     14085
weighted avg       0.86      0.67      0.74     14085



### Random Forest model

In [104]:

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
         n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))


Accuracy: 0.9107561235356763
Precision: 1.0
Recall: 0.000794912559618442
F1: 0.0015885623510722795


### Gradient Boosting

In [105]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

gb_pipeline.fit(X_train, y_train)

y_pred = gb_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9106851260205893
Precision: 0.5
Recall: 0.0023847376788553257
F1: 0.004746835443037975
              precision    recall  f1-score   support

           0       0.91      1.00      0.95     12827
           1       0.50      0.00      0.00      1258

    accuracy                           0.91     14085
   macro avg       0.71      0.50      0.48     14085
weighted avg       0.87      0.91      0.87     14085

